# Food recommendations from blood tests

This notebook shows how **biofoodrec** works, using the demo patient from the Biolabs website.

You give it blood test results. It tells you:

1. which results are high or low
2. which foods fit this person best, and which to limit
3. which nutrients to eat more of or less of
4. why each food got its place

**How to run it:** click a grey code box and press **Shift + Enter**. Go from top to bottom the first time.
After that you can change any number and run that box again.

## 1. Start

In [1]:
import json

import numpy as np
import pandas as pd
from biofoodrec import FoodRecommender, Tuning, sample_patient

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 70)

BLUE, RED = "#2a78d6", "#e34948"

def bars(table, *columns, axis=0):
    """Small bars inside a table: blue to the right of zero, red to the left."""
    return (table.style.bar(subset=list(columns), axis=axis, align="zero", color=[RED, BLUE], height=60, width=80)
            .format(precision=2))

rec = FoodRecommender()
ref = rec.reference
print(len(ref.biochemicals), "blood tests |", len(ref.foods), "foods |", len(ref.nutrients), "nutrients")

58 blood tests | 150 foods | 75 nutrients


The Biolabs data is inside the package, so there is nothing to download.
The default settings measure every test the same way and keep four safety rules on.
Sections 10 and 11 explain both.

## 2. Load a patient from JSON

This is the demo patient the Biolabs website loads: a 46 year old woman with 4 visits.
Here are the first 3 results of her last visit, exactly as they sit in the JSON:

In [2]:
patient = sample_patient()
print("Visits:", len(patient["biometrics_entries"]), "| sex:", patient["user"]["gender"])
patient["biometrics_entries"][-1]["biometrics"][:3]

Visits: 4 | sex: female


[{'id': 1, 'name': 'Blood Glucose', 'unit': 'mg/dL', 'value': 102.4},
 {'id': 2, 'name': 'Hemoglobin A1c', 'unit': '%', 'value': 5.4},
 {'id': 3, 'name': 'Glycated Albumin', 'unit': '%', 'value': 13.5}]

Give the whole JSON to the recommender. It takes the newest result of every test.

In [3]:
result = rec.recommend(patient)
result

<Recommendation: 150 foods, best: Parsley, Flax Seed, Cabbage>

These JSON shapes work too. You can pass a Python object, a JSON string, or a path to a file:

```text
{"Blood Glucose": 130, "Vitamin D": 14}
{"Blood Glucose": {"value": 130}}
[{"id": 1, "value": 130}]          (a list, by id 1 to 58 or by name)
{"data": {"updatedBiochemicalsData": [...], "unExpiredBiometricsData": [...]}}
{"user": {"gender": "female"}, "biometrics_entries": [...]}
```

The first three do not say the sex, so add `sex="female"` or `sex="male"`, like `rec.recommend(labs, sex="female")`.

The backend's foods score request `{"data": [{"id": 1, "value": 2.4}]}` holds values that are already scaled, so it goes to `rec.score(...)` instead.

## 3. Lab results

**How far off** tells you where a result sits. 0 is the middle of the healthy range. 1 or minus 1 is the edge.
More than 1 means outside the range. Blue bars are above the middle, red bars are below.
Results outside the range come first.

**Changes advice** says whether the result changes the food list. Only results outside the healthy range do (section 11).

In [4]:
bars(result.labs().head(15), "How far off")

,Result,Healthy range,Status,How far off,Changes advice
Biochemical,,,,,
C-Reactive Protein,4 mg/L,0 to 3,High,1.67,Yes
Gamma-Glutamyl Transferase,43 U/L,5 to 36,High,1.45,Yes
Thyroid-Stimulating Hormone (TSH),4.4 mIU/L,0.4 to 4,High,1.22,Yes
Uric Acid,6.3 mg/dL,2.4 to 6,High,1.17,Yes
Blood Glucose,102.4 mg/dL,70 to 100,High,1.16,Yes
Vitamin D,18.4 ng/mL,20 to 50,Low,-1.11,Yes
Iron,47.8 µg/dL,50 to 170,Low,-1.04,Yes
Total Cholesterol,198.3 mg/dL,125 to 200,"Normal, upper side",0.95,No
Progesterone,1.8 ng/mL,0.1 to 25,"Normal, lower side",-0.86,No


## 4. The short report

One text you can read or send. It names the results that are out of range, the nutrients to eat more and
less of, and the best and worst foods.

In [5]:
print(result.summary())

58 lab results, 7 outside the healthy range.
High: C-Reactive Protein 4 mg/L (healthy 0 to 3); Gamma-Glutamyl Transferase 43 U/L (healthy 5 to 36); Thyroid-Stimulating Hormone (TSH) 4.4 mIU/L (healthy 0.4 to 4); Uric Acid 6.3 mg/dL (healthy 2.4 to 6); Blood Glucose 102.4 mg/dL (healthy 70 to 100)
Low: Vitamin D 18.4 ng/mL (healthy 20 to 50); Iron 47.8 µg/dL (healthy 50 to 170)

Eat more: Vitamin C, Antioxidants, Fiber, Omega-3 Fatty Acids, Polyphenols, Magnesium
Eat less: Proteins, Caffeine, Alcohol, Fructose, Trans Fatty Acids, Oxalates

Best 10 foods (score out of 100):
  1. Parsley  88.1  helps lower C-Reactive Protein and Blood Glucose
  2. Flax Seed  85.9  helps lower C-Reactive Protein and Blood Glucose
  3. Cabbage  85.8  helps lower C-Reactive Protein and Blood Glucose
  4. Lettuce  85.8  helps lower C-Reactive Protein and Blood Glucose
  5. Strawberry  85.5  helps lower C-Reactive Protein and Uric Acid
  6. Mint  85.5  helps lower C-Reactive Protein and Blood Glucose
  7. Chia

## 5. Foods

**Score** is out of 100. A food gets points three ways: what it does to each blood test directly, what its
nutrients do, and its general quality (nutriscore). 100 would mean the best on all three.
**Why** names the tests the food helps most, starting with the ones out of range.

In [6]:
result.foods(20)

,Food,Group,Score,Why
Rank,,,,
1,Parsley,Vegetable,88.1,helps lower C-Reactive Protein and Blood Glucose
2,Flax Seed,Seed,85.9,helps lower C-Reactive Protein and Blood Glucose
3,Cabbage,Vegetable,85.8,helps lower C-Reactive Protein and Blood Glucose
4,Lettuce,Vegetable,85.8,helps lower C-Reactive Protein and Blood Glucose
5,Strawberry,Fruit,85.5,helps lower C-Reactive Protein and Uric Acid
6,Mint,Vegetable,85.5,helps lower C-Reactive Protein and Blood Glucose
7,Chia Seed,Seed,85.3,helps lower C-Reactive Protein and Blood Glucose
8,Zucchini,Vegetable,83.4,helps lower C-Reactive Protein and Gamma-Glutamyl Transferase
9,Cucumber,Vegetable,83.3,helps lower C-Reactive Protein and Gamma-Glutamyl Transferase


Foods to limit, worst first:

In [7]:
result.limit(10)

,Food,Group,Score,Why
Rank,,,,
150,Brownies,Nosh,9.4,raises Uric Acid and Blood Glucose
149,Pastries,Nosh,15.7,raises Uric Acid and Gamma-Glutamyl Transferase
148,Muffins,Nosh,16.0,raises Uric Acid and Blood Glucose
147,Cakes,Nosh,16.4,raises Blood Glucose and C-Reactive Protein
146,Cookies,Nosh,17.5,raises Uric Acid and Blood Glucose
145,Donuts,Nosh,17.5,raises Blood Glucose and C-Reactive Protein
144,Cereals,Nosh,18.5,raises Uric Acid and Gamma-Glutamyl Transferase
143,Croissants,Nosh,18.9,raises Uric Acid and Gamma-Glutamyl Transferase
142,Potato Chips,Nosh,20.7,raises Uric Acid and Gamma-Glutamyl Transferase


## 6. Nutrients

**Need** goes from 100 (eat much more) to minus 100 (eat much less).
**Found in** lists the richest foods. For nutrients to eat more of, it only lists foods that also suit this person.

In [8]:
nutrients = result.nutrients()
bars(nutrients.head(12), "Need")

,Advice,Need,Group,Found in
Nutrient,,,,
Vitamin C,Eat more,100,Vitamins,"Guava, Thyme, Parsley"
Antioxidants,Eat more,74,Other,"Mint, Thyme, Cauliflower"
Fiber,Eat more,63,Other,"Oregano, Chia Seed, Flax Seed"
Omega-3 Fatty Acids,Eat more,59,Fatty Acids,"Flax Seed, Chia Seed, Mackerel"
Polyphenols,Eat more,53,Phytochemicals,"Guacamole, Shiitake Mushrooms, Portobello Mushrooms"
Magnesium,Eat more,50,Minerals,"Pumpkin Seed, Flax Seed, Sesame Seed"
Prebiotics,Eat more,36,Other,"Oregano, Thyme, Rosemary"
Flavonoids,Eat more,27,Phytochemicals,"Oregano, Mint, Thyme"
Iron,Eat more,24,Minerals,"Oregano, Turmeric, Thyme"


Nutrients to eat less of, strongest first:

In [9]:
bars(nutrients[nutrients.Advice == "Eat less"].sort_values("Need").head(10), "Need")

,Advice,Need,Group,Found in
Nutrient,,,,
Proteins,Eat less,-87,Macronutrients,"Bacon, Beef Jerky, Chicken"
Caffeine,Eat less,-80,Other,"Tea, Coffee, Coca Cola"
Alcohol,Eat less,-78,Other,Beer
Fructose,Eat less,-73,Sugars,"Brownies, Banana, Cakes"
Trans Fatty Acids,Eat less,-72,Fatty Acids,"Ghee, Butter, Pepperoni"
Oxalates,Eat less,-58,Other,"Celery, Cassava, Beets"
Phytic Acid,Eat less,-56,Other,"Crackers, Yams, Potatoes"
Carbohydrates,Eat less,-55,Macronutrients,"Cereals, Popcorn, Barley"
Vitamin B12,Eat less,-53,Vitamins,"Mussels, Oysters, Mackerel"


## 7. Why a food got its place

Each row is one blood test that changes the advice.

1. **Your result**: high or low
2. **This food**: does the food raise or lower that test
3. **Direct** and **Through nutrients**: the two ways the food acts, in score points
4. **Points**: how many points out of 100 this test gives the food. Positive pushes it up the list, negative pushes it down.

A food's score is its starting points (its general quality) plus all these points, exactly.
Change the food name and run the box again.

In [10]:
bars(result.explain("Flax Seed").head(12), "Points")

,Your result,This food,Direct,Through nutrients,Points,Verdict
Biochemical,,,,,,
C-Reactive Protein,High,lowers it,4.59,0.53,5.11,Good
Blood Glucose,High,lowers it,2.59,1.36,3.96,Good
Uric Acid,High,lowers it,2.01,0.01,2.02,Good
Vitamin D,Low,raises it,0.00,1.46,1.46,Good
Iron,Low,raises it,-0.89,0.63,-0.26,Bad
Gamma-Glutamyl Transferase,High,raises it,2.50,-2.62,-0.12,Bad
Thyroid-Stimulating Hormone (TSH),High,raises it,1.68,-1.65,0.03,Good


In [11]:
bars(result.explain("Banana").head(12), "Points")

,Your result,This food,Direct,Through nutrients,Points,Verdict
Biochemical,,,,,,
Blood Glucose,High,raises it,-3.19,-2.25,-5.44,Bad
Thyroid-Stimulating Hormone (TSH),High,raises it,-2.52,-0.35,-2.87,Bad
Iron,Low,raises it,0.89,0.14,1.03,Good
Uric Acid,High,raises it,0.00,-0.93,-0.93,Bad
Gamma-Glutamyl Transferase,High,raises it,0.00,-0.87,-0.87,Bad
C-Reactive Protein,High,raises it,0.00,-0.26,-0.26,Bad
Vitamin D,Low,raises it,0.00,0.09,0.09,Good


## 8. Change a result and everything updates

Say the next visit shows high glucose and good vitamin D. Change those two numbers.
The lab table, the foods, the nutrients and the report all update.

In [12]:
after = result.update({"Blood Glucose": 180, "Vitamin D": 45})
after.labs().head(8)

,Result,Healthy range,Status,How far off,Changes advice
Biochemical,,,,,
Blood Glucose,180 mg/dL,70 to 100,High,6.33,Yes
C-Reactive Protein,4 mg/L,0 to 3,High,1.67,Yes
Gamma-Glutamyl Transferase,43 U/L,5 to 36,High,1.45,Yes
Thyroid-Stimulating Hormone (TSH),4.4 mIU/L,0.4 to 4,High,1.22,Yes
Uric Acid,6.3 mg/dL,2.4 to 6,High,1.17,Yes
Iron,47.8 µg/dL,50 to 170,Low,-1.04,Yes
Total Cholesterol,198.3 mg/dL,125 to 200,"Normal, upper side",0.95,No
Progesterone,1.8 ng/mL,0.1 to 25,"Normal, lower side",-0.86,No


**Moved** shows how many places each food went up (positive) or down (negative).

In [13]:
after.compare(result)

,Rank,Was,Moved,Score
Food,,,,
Parsley,1,1,0,89.2
Mint,2,6,4,88.4
Flax Seed,3,2,-1,87.8
Chia Seed,4,7,3,86.7
Lettuce,5,4,-1,86.7
Cabbage,6,3,-3,86.5
Oregano,7,13,6,85.9
Thyme,8,14,6,85.7
Broccoli Sprouts,9,11,2,84.8


In [14]:
print(after.summary())

58 lab results, 6 outside the healthy range.
High: Blood Glucose 180 mg/dL (healthy 70 to 100); C-Reactive Protein 4 mg/L (healthy 0 to 3); Gamma-Glutamyl Transferase 43 U/L (healthy 5 to 36); Thyroid-Stimulating Hormone (TSH) 4.4 mIU/L (healthy 0.4 to 4); Uric Acid 6.3 mg/dL (healthy 2.4 to 6)
Low: Iron 47.8 µg/dL (healthy 50 to 170)

Eat more: Fiber, Antioxidants, Polyphenols, Vitamin C, Omega-3 Fatty Acids, Magnesium
Eat less: Fructose, Carbohydrates, Glucose, Sucrose, Alcohol, Proteins

Best 10 foods (score out of 100):
  1. Parsley  89.2  helps lower Blood Glucose and C-Reactive Protein
  2. Mint  88.4  helps lower Blood Glucose and C-Reactive Protein
  3. Flax Seed  87.8  helps lower Blood Glucose and C-Reactive Protein
  4. Chia Seed  86.7  helps lower Blood Glucose and C-Reactive Protein
  5. Lettuce  86.7  helps lower Blood Glucose and C-Reactive Protein
  6. Cabbage  86.5  helps lower Blood Glucose and C-Reactive Protein
  7. Oregano  85.9  helps lower Blood Glucose and C-Rea

## 9. Under the hood: the matrix math

This part is for researchers. The whole recommendation is three matrix multiplications.

1. **deviation**: one number per blood test (58 numbers)
2. **drive** = deviation × importance × minus 1. A high test pushes down, a low test pushes up.
3. **direct** = drive @ food_coefficients. (58) @ (58 × 150) gives one number per food.
4. **need** = drive @ nutrient_coefficients. (58) @ (58 × 75) gives one number per nutrient.
5. **through** = food_nutrients @ need. (150 × 75) @ (75) gives one number per food.
6. **score** = direct scaled to 0..1, plus through scaled to 0..1, plus nutriscore.

Every coefficient is bias + weight. Bias is the direction: 1 means the food raises the test, minus 1 means it lowers it.
Weight is the strength. Both tables were written by hand in the research notebooks.

In [15]:
m = result.matrices()
pd.DataFrame({"shape": {name: str(x.shape) for name, x in m.items()}})

,shape
deviation,"(58,)"
drive,"(58,)"
food_coefficients,"(58, 150)"
nutrient_coefficients,"(58, 75)"
food_nutrients,"(150, 75)"
direct,"(150,)"
need_before_guard,"(75,)"
need,"(75,)"
guarded_foods,"(150,)"
through_nutrients,"(150,)"


The deviation vector for this patient (tests sitting exactly in the middle are left out):

In [16]:
deviation = m["deviation"]
bars(deviation[deviation != 0].sort_values().to_frame(), "deviation")

,deviation
name,
Vitamin D,-1.11
Iron,-1.04
Blood Glucose,1.16
Uric Acid,1.17
Thyroid-Stimulating Hormone (TSH),1.22
Gamma-Glutamyl Transferase,1.45
C-Reactive Protein,1.67


A corner of the food coefficient matrix. Positive means the food raises the test:

In [17]:
m["food_coefficients"].loc[["Blood Glucose", "LDL Cholesterol", "Vitamin D", "C-Reactive Protein"],
                           ["Flax Seed", "Salmon", "Banana", "Donuts"]]

name,Flax Seed,Salmon,Banana,Donuts
name,,,,
Blood Glucose,-1.3,-1.0,1.6,1.7
LDL Cholesterol,-1.4,-1.5,0.9,0.8
Vitamin D,0.0,2.0,0.0,0.0
C-Reactive Protein,-1.6,-1.7,0.0,1.7


Now do the math by hand and check that it gives the same scores as the package:

In [18]:
direct = m["drive"] @ m["food_coefficients"]       # 150 numbers, one per food
need = m["drive"] @ m["nutrient_coefficients"]     # 75 numbers, one per nutrient
through = m["food_nutrients"] @ need               # 150 numbers, one per food
through[m["guarded_foods"] & (through > 0)] = 0  # the food guard (section 11)

scale = lambda x: (x - x.min()) / (x.max() - x.min())
score = scale(direct) + scale(through) + m["parts"]["nutriscore"]

print("Same as the package:", np.allclose(score, m["score"]))
print("Points add up too:   ", np.allclose(m["points"].sum() + m["base_points"], 100 * m["score"] / 3))
score.sort_values(ascending=False).head(10).round(3)

Same as the package: True
Points add up too:    True


name
Parsley         2.642
Flax Seed       2.577
Cabbage         2.574
Lettuce         2.573
Strawberry      2.565
Mint            2.564
Chia Seed       2.560
Zucchini        2.501
Cucumber        2.499
Bell Peppers    2.494
dtype: float64

The pull matrix (58 × 150) says how much each test moves each food. Here are the tests out of range,
against the 8 best and the 4 worst foods. Blue pushes the food up, red pushes it down.

In [19]:
off_range = result.labs().query("Status in ['High', 'Low']").index
columns = list(result.table.index[:8]) + list(result.table.index[-4:])
bars(m["pull"].loc[off_range, columns], *columns, axis=None)

name,Parsley,Flax Seed,Cabbage,Lettuce,Strawberry,Mint,Chia Seed,Zucchini,Cakes,Muffins,Pastries,Brownies
Biochemical,,,,,,,,,,,,
C-Reactive Protein,4.89,3.47,2.30,2.23,2.58,5.44,3.31,2.23,-4.88,-5.40,-5.56,-6.00
Gamma-Glutamyl Transferase,-0.42,-2.55,-0.66,-0.95,-0.42,-0.75,-2.33,0.92,-6.10,-7.13,-7.72,-7.80
Thyroid-Stimulating Hormone (TSH),-0.16,-1.54,0.21,1.01,-0.51,-0.45,-1.97,0.76,-2.89,-3.69,-3.65,-4.07
Uric Acid,2.31,1.19,1.21,0.53,0.77,1.93,1.12,0.83,-7.28,-11.99,-13.54,-13.32
Blood Glucose,3.63,3.59,0.90,1.45,0.09,4.09,3.17,1.02,-7.52,-7.81,-5.88,-8.62
Vitamin D,1.75,2.23,-0.21,0.37,0.08,0.65,2.46,0.13,0.40,0.52,0.51,0.76
Iron,0.58,0.44,-0.34,-0.86,0.98,-0.86,0.21,-0.77,0.32,0.30,0.19,0.34


How much did this patient's results change the list? The research notebook measured how far each food
moved compared with a list sorted by nutriscore alone:

In [20]:
print(f"Each food moved {result.moved_places:.0f} places on average")

Each food moved 22 places on average


## 10. Which scaling to use

The live Biolabs backend (**legacy**) measures distance outside the healthy range in the test's own unit.
So a result just past the edge can look normal, and tests with big units count more.
The research notebook added 1 at the edge (**original**). **linear** measures every test in half ranges,
the same way inside and outside the range. This notebook uses linear.

Look at HbA1c 5.8 (the healthy max is 5.6). Legacy gives it 0.2, so the backend calls it normal.

In [21]:
border = {"Hemoglobin A1c": 5.8, "Thyroid-Stimulating Hormone (TSH)": 4.3,
          "Gamma-Glutamyl Transferase": 43, "Blood Glucose": 102}
table = pd.DataFrame({name: rec.deviations(border, "female", Tuning(scaling=name, cap=None)).deviation
                      for name in ["legacy", "original", "linear"]}).round(2)
table.insert(0, "Result", pd.Series(border))
table

,Result,legacy,original,linear
biochemical,,,,
Hemoglobin A1c,5.8,0.2,1.2,1.25
Thyroid-Stimulating Hormone (TSH),4.3,0.3,1.3,1.17
Gamma-Glutamyl Transferase,43.0,7.0,8.0,1.45
Blood Glucose,102.0,2.0,3.0,1.13


`Tuning.legacy()` gives exactly the numbers the live backend gives today, if you need to match it.
It has no safety guards, so use it only to compare.

## 11. Safety rules

The tables hold many true facts, like "alcohol lowers vitamin B1". Read as a treatment, that fact would mean
"drink alcohol when vitamin B1 is high". Four rules stop this, and all are on by default:

1. **Only results outside the healthy range act.** A normal result never changes the advice.
2. **One sided tests act on one side only.** A low LDL or CRP, or a high HDL, never asks for a diet change.
3. **limit_only**: alcohol, trans fats, saturated fats, cholesterol, sodium, free sugars, caffeine and the anti
   nutrients can be advised down, never up.
4. **food_guard**: foods with a nutriscore of 0.3 or less (bacon, sodas, pastries, burgers) can move down the list,
   never up.

First, a healthy LDL of 60. The live backend's settings push it toward the middle of the range and recommend butter.
The defaults leave the list alone:

In [22]:
for name, tuning in (("Live backend (legacy)", Tuning.legacy()), ("Default", Tuning())):
    print(f"{name}: best 4 foods", ", ".join(rec.recommend({"LDL Cholesterol": 60}, sex="female", tuning=tuning).table.index[:4]))

Live backend (legacy): best 4 foods Ghee, Butter, Crab, Shrimp
Default: best 4 foods Strawberry, Blueberry, Raspberry, Spinach


Now a kidney patient with high potassium, first with rules 3 and 4 switched off, then with the defaults:

In [23]:
kidney = {"Creatinine": 2.2, "Blood Urea Nitrogen": 40, "Potassium": 5.9, "Phosphate": 5.5}
guards_off = Tuning(limit_only=[], food_guard=None)
for name, tuning in (("Rules 3 and 4 off", guards_off), ("All rules on (default)", Tuning())):
    advice = rec.recommend(kidney, sex="male", tuning=tuning).nutrients()
    print(f"{name}: eat more", ", ".join(advice[advice.Advice == "Eat more"].index))

Rules 3 and 4 off: eat more Water, Calcium, Carbohydrates, Trans Fatty Acids, Alcohol, Oxalates, Sodium, Tannins, Phytic Acid, Fructose
All rules on (default): eat more Water, Calcium, Carbohydrates


With rules 3 and 4 off, the advice includes trans fats, alcohol and oxalates. With them on, it does not.
The package tests check this for every one of the 58 tests, far above and far below the healthy range, for women
and men. The cells behind problems like this are listed in `review/table_review.csv` for a nutritionist to check.

## 12. Personal settings

An admin keeps one set of settings for everyone. Each person can get a small patch on top.
Only the patch needs to be saved for that person.

In [24]:
patch = {
    "importance": {"Blood Glucose": 2.0},   # glucose counts double for this person
    "ranges": {"Vitamin D": [30, 60]},      # their own target range
    "exclude_groups": ["seafood"],          # seafood allergy
    "exclude_foods": ["Whole Milk"],        # dislikes
}
personal = rec.recommend(patient, tuning=Tuning().update(patch))
personal.compare(result)

,Rank,Was,Moved,Score
Food,,,,
Parsley,1,1,0,87.5
Flax Seed,2,2,0,85.9
Chia Seed,3,7,4,85.3
Mint,4,6,2,85.2
Lettuce,5,4,-1,84.4
Cabbage,6,3,-3,84.1
Oregano,7,13,6,83.6
Thyme,8,14,6,83.3
Strawberry,9,5,-4,83.2


In [25]:
print(json.dumps(patch, indent=2))   # what you would save for this person

{
  "importance": {
    "Blood Glucose": 2.0
  },
  "ranges": {
    "Vitamin D": [
      30,
      60
    ]
  },
  "exclude_groups": [
    "seafood"
  ],
  "exclude_foods": [
    "Whole Milk"
  ]
}


## 13. Try the settings with sliders

Drag a slider and the table updates. It shows the new top 12 against the list from section 5.

In [ ]:
import ipywidgets as w

style = {"description_width": "140px"}
slider = lambda label, value=1.0, top=3.0: w.FloatSlider(value=value, min=0, max=top, step=0.1, description=label,
                                                         style=style, layout=w.Layout(width="440px"))

def show(scaling, act_in_range, food_mix, nutrient_mix, nutriscore_mix, bias_scale, weight_scale, glucose, crp):
    tuning = Tuning(scaling=scaling, act_in_range=act_in_range, cap=4.0 if scaling == "linear" else None,
                    food_mix=food_mix, nutrient_mix=nutrient_mix, nutriscore_mix=nutriscore_mix,
                    bias_scale=bias_scale, weight_scale=weight_scale,
                    importance={"Blood Glucose": glucose, "C-Reactive Protein": crp})
    display(rec.recommend(patient, tuning=tuning).compare(result, n=12))

w.interact(show,
           scaling=w.ToggleButtons(options=["linear", "original", "legacy"], description="scaling", style=style),
           act_in_range=w.Checkbox(value=False, description="normal results act too"),
           food_mix=slider("food (direct)"), nutrient_mix=slider("food (nutrients)"), nutriscore_mix=slider("nutriscore"),
           bias_scale=slider("bias"), weight_scale=slider("weight"),
           glucose=slider("glucose importance"), crp=slider("CRP importance"));

## 14. Send it to your app

`to_json()` gives everything as JSON, ready for an API or a frontend.
Here is the start of it:

In [27]:
print(result.to_json()[:1200])

{
  "labs": [
    {
      "biochemical": "C-Reactive Protein",
      "result": "4 mg/L",
      "healthy_range": "0 to 3",
      "status": "High",
      "how_far_off": 1.67,
      "changes_advice": "Yes"
    },
    {
      "biochemical": "Gamma-Glutamyl Transferase",
      "result": "43 U/L",
      "healthy_range": "5 to 36",
      "status": "High",
      "how_far_off": 1.45,
      "changes_advice": "Yes"
    },
    {
      "biochemical": "Thyroid-Stimulating Hormone (TSH)",
      "result": "4.4 mIU/L",
      "healthy_range": "0.4 to 4",
      "status": "High",
      "how_far_off": 1.22,
      "changes_advice": "Yes"
    },
    {
      "biochemical": "Uric Acid",
      "result": "6.3 mg/dL",
      "healthy_range": "2.4 to 6",
      "status": "High",
      "how_far_off": 1.17,
      "changes_advice": "Yes"
    },
    {
      "biochemical": "Blood Glucose",
      "result": "102.4 mg/dL",
      "healthy_range": "70 to 100",
      "status": "High",
      "how_far_off": 1.16,
      "changes_

## 15. Use your own data

```python
from biofoodrec import FoodRecommender, ReferenceData

ref = ReferenceData.from_csv_dir("my_folder")                                 # all 7 files
ref = ReferenceData.from_csv_dir("my_folder", food_weight="new_weights.csv")  # swap one file
rec = FoodRecommender(ref)
```

Tables are matched by name, so the column order does not matter.
If a test or a food is missing, the error message names it.